# Layer 4: Orderbook Dynamics
O1–O5: update rates, BBO spread, delta compression, depth activity, level lifetime.

In [ ]:
import os, numpy as np, pandas as pd

DATA_DIR    = os.environ.get("DATA_DIR",    "/data/parquet")
REPORTS_DIR = os.environ.get("REPORTS_DIR", "/data/reports")
os.makedirs(REPORTS_DIR, exist_ok=True)

from mda.loader import load_orderbook, load_orderbook_updates
from mda.layer4.bbo         import compute_bbo, spread_stats
from mda.layer4.update_rates import compute_update_rate_by_depth, compute_delta_compression_ratio
from mda.layer4.levels       import compute_level_lifetimes_polars, level_lifetime_stats
from mda.plots.charts import (
    plot_update_rate_by_depth, plot_bbo_spread_depth,
    plot_delta_compression_ts, plot_book_activity_heatmap,
    plot_level_lifetime_hist, save_figure,
)
print("imports OK")

In [ ]:
# BBO only (level==0) — much cheaper memory-wise
ob_bbo = load_orderbook(DATA_DIR, bbo_only=True)
print(f"BBO rows: {len(ob_bbo):,} | exchanges: {ob_bbo['exchange'].unique().tolist()}")

In [ ]:
bbo = compute_bbo(ob_bbo)
s   = spread_stats(bbo)
print("BBO spread stats (bps):")
print(s.to_string(index=False))

In [ ]:
# Annotate bbo with ts_dt for the chart
import pandas as pd
bbo["ts_dt"] = pd.to_datetime(bbo["received_at"], utc=True, format="mixed")
fig = plot_bbo_spread_depth(bbo)
fig.show()
save_figure(fig, "O2_bbo_spread_depth", REPORTS_DIR)

In [ ]:
# Load full orderbook (all levels) per exchange to manage memory
# Compute update rates across all exchanges at once
ob_full = load_orderbook(DATA_DIR)
print(f"Full OB rows: {len(ob_full):,}")

In [ ]:
update_rate = compute_update_rate_by_depth(ob_full)
print("Update rate by depth (top 10 levels per exchange):")
print(update_rate[update_rate["level"] < 10].to_string(index=False))

In [ ]:
fig = plot_update_rate_by_depth(update_rate[update_rate["level"] < 20])
fig.show()
save_figure(fig, "O1_update_rate_by_depth", REPORTS_DIR)

In [ ]:
for exch in ob_full["exchange"].unique():
    fig = plot_book_activity_heatmap(update_rate, exchange=exch)
    fig.show()
    save_figure(fig, f"O4_book_activity_{exch}", REPORTS_DIR)

In [ ]:
ob_updates = load_orderbook_updates(DATA_DIR)
delta = compute_delta_compression_ratio(ob_full, ob_updates)
fig = plot_delta_compression_ts(delta)
fig.show()
save_figure(fig, "O3_delta_compression_ts", REPORTS_DIR)

In [ ]:
# Level lifetime via Polars (memory-efficient streaming)
lifetime_parts = []
for exch in ob_full["exchange"].unique():
    try:
        lt = compute_level_lifetimes_polars(DATA_DIR, exchange=exch, symbol="BTCUSDT")
        if lt is not None and len(lt) > 0:
            lifetime_parts.append(lt)
            print(f"{exch}: {len(lt):,} levels")
    except Exception as e:
        print(f"{exch}: skipped ({e})")

if lifetime_parts:
    import pandas as pd
    lifetimes = pd.concat(lifetime_parts, ignore_index=True)
    print(level_lifetime_stats(lifetimes).to_string(index=False))
    fig = plot_level_lifetime_hist(lifetimes)
    fig.show()
    save_figure(fig, "O5_level_lifetime_hist", REPORTS_DIR)